# Chapter 9 - Lab 1: <font color='blue'>Sequential Claims Pipeline with Compliance Guardrail</font>

**<font color='purple'>Goal</font>**:
Build a **7-agent sequential pipeline** that processes a real auto-collision claim from First Notice of Loss to final compliance review. The pipeline is the **Sequential Pipeline with Guardrails** architecture from Chapter 7 (§2.1) applied to insurance.

Seven agents pass control via explicit handoffs, share data through a typed `Context` object, and write to an audit trail at every step. The seventh agent — Compliance — has *veto power* and can block a decision that fails regulatory checks.

By the end you will see an entire claims workflow run end-to-end on a sample claim, with every agent's contribution recorded for audit.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex** — `FunctionAgent`, `AgentWorkflow`, `Context`.
* **OpenAI** `gpt-4o`.
* **Pydantic** — typed contracts for the claim record and helpers from `common.py`.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 9 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%209/common.py',
        'common.py',
    )

from common import (
    ClaimType, ClaimStatus,
    SAMPLE_CLAIM, SAMPLE_POLICY, SAMPLE_DOCUMENTS,
    generate_claim_id, log_audit, days_between,
)

print('Common helpers loaded.')

## 4. Define the seven pipeline tools

Each agent has a dedicated async tool that reads from and writes to the shared `Context`. This is the same shared-state pattern from Chapter 7 — but here every write also calls `log_audit` to record what happened, when, and which agent did it. The audit log is the most important artifact for compliance: any decision must be reconstructable from it.

In [ ]:
from llama_index.core.workflow import Context

def classify_claim_type(description: str) -> ClaimType:
    d = description.lower()
    if any(w in d for w in ['vehicle', 'car', 'truck', 'collision', 'bumper']):
        return ClaimType.AUTO
    if any(w in d for w in ['house', 'roof', 'flood', 'fire', 'property']):
        return ClaimType.PROPERTY
    if any(w in d for w in ['hospital', 'medical', 'surgery', 'injury']):
        return ClaimType.HEALTH
    return ClaimType.LIABILITY


async def process_fnol(ctx: Context, claim_data: dict) -> str:
    if isinstance(claim_data, str):  # some models pass the JSON as a string
        claim_data = json.loads(claim_data)
    claim_id = generate_claim_id()
    ct = classify_claim_type(claim_data.get('description', ''))
    claim = {
        'claim_id': claim_id,
        'policyholder_name': claim_data.get('name', ''),
        'policy_number': claim_data.get('policy_number', ''),
        'claim_type': ct.value,
        'date_of_loss': claim_data.get('date_of_loss', ''),
        'description': claim_data.get('description', ''),
        'status': 'received', 'audit_log': [],
    }
    state = await ctx.store.get('state')
    state['claim'] = claim
    log_audit(state, 'FNOLAgent', 'Claim created', f'ID: {claim_id}, Type: {ct.value}')
    await ctx.store.set('state', state)
    return f'FNOL processed. Claim {claim_id}.'


async def parse_documents(ctx: Context) -> str:
    state = await ctx.store.get('state')
    claim = state['claim']
    docs = state.get('submitted_documents', [])
    parsed, damage_parts = [], []
    for d in docs:
        if d.get('type') == 'photo':
            desc = d.get('description', '')
            parsed.append({'type': 'damage_photo', 'damage_description': desc})
            damage_parts.append(desc)
        elif d.get('type') == 'claim_form':
            parsed.append({'type': 'claim_form', 'fields': d.get('fields', {})})
        elif d.get('type') == 'police_report':
            parsed.append({'type': 'police_report', 'summary': d.get('summary', '')})
    claim['parsed_documents'] = parsed
    claim['damage_description'] = ' '.join(damage_parts)
    claim['status'] = 'parsing'
    log_audit(state, 'DocumentParser', f'Parsed {len(parsed)} docs')
    await ctx.store.set('state', state)
    return f'Parsed {len(parsed)} documents.'

**Coverage Agent.** Checks whether the loss falls within the policy period and is not excluded. Writes back the coverage flag, limit, deductible, and reasoning.

In [ ]:
def coverage_evidence(claim: dict, policy: dict) -> tuple[list[str], bool]:
    """Return supported denial codes and whether basic lab evidence is complete.

    Exclusion matching is a simplified lab rule, not a legal coverage assessment.
    """
    from datetime import date

    try:
        loss = date.fromisoformat(claim["date_of_loss"])
        start = date.fromisoformat(policy["inception_date"])
        end = date.fromisoformat(policy["expiration_date"])
    except (KeyError, TypeError, ValueError):
        return [], False
    if start > end:
        return [], False
    codes = []
    if not start <= loss <= end:
        codes.append("OUTSIDE_POLICY_PERIOD")
    exclusions = policy.get("exclusions")
    if not isinstance(exclusions, list) or not all(isinstance(x, str) and x.strip() for x in exclusions):
        return codes, False
    description = claim.get("description", "")
    if not isinstance(description, str) or not description.strip():
        return codes, False
    if any(x.lower() in description.lower() for x in exclusions):
        codes.append("POLICY_EXCLUSION")
    return codes, True

async def verify_coverage(ctx: Context) -> str:
    """Check policy coverage for this claim.

    Used by: CoverageAgent.
    Reads: claim, policy from state.
    Writes: coverage fields to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]
    policy = state.get("policy", {})

    # Check basic coverage
    reason_codes, evidence_complete = coverage_evidence(claim, policy)
    is_covered = evidence_complete and not reason_codes
    exclusions = []
    reasoning_parts = []
    if not evidence_complete:
        reasoning_parts.append("Coverage evidence is incomplete or invalid; human review required.")

    # Check policy period
    dol = claim.get("date_of_loss", "")
    inception = policy.get("inception_date", "")
    expiration = policy.get("expiration_date", "")

    if "OUTSIDE_POLICY_PERIOD" in reason_codes:
        reasoning_parts.append(
            f"Date of loss {dol} outside policy period "
            f"({inception} to {expiration})."
        )

    # Check exclusions
    policy_exclusions = policy.get("exclusions", [])
    description = claim.get("description", "")
    for exclusion in (policy_exclusions if isinstance(policy_exclusions, list) else []):
        if not isinstance(exclusion, str) or not exclusion.strip():
            continue
        if isinstance(description, str) and exclusion.lower() in description.lower():
            exclusions.append(exclusion)

    if exclusions:
        reasoning_parts.append(
            f"Exclusions found: {exclusions}"
        )

    if is_covered and not exclusions:
        reasoning_parts.append(
            f"Event covered under {policy.get('coverage_sections', 'comprehensive')} "
            f"coverage. Limit: ${policy.get('coverage_limit', 0):,.0f}. "
            f"Deductible: ${policy.get('deductible', 0):,.0f}."
        )

    claim["coverage_verified"] = is_covered and not exclusions
    claim["applicable_limit"] = policy.get("coverage_limit", 0)
    claim["deductible"] = policy.get("deductible", 0)
    claim["exclusions_found"] = exclusions
    claim["coverage_reasoning"] = " ".join(reasoning_parts)
    claim["coverage_reason_codes"] = reason_codes
    claim["coverage_evidence_complete"] = evidence_complete
    claim["status"] = "coverage_check"

    log_audit(state, "CoverageAgent",
              f"Coverage: {'verified' if claim['coverage_verified'] else 'denied'}",
              claim["coverage_reasoning"])
    await ctx.store.set("state", state)
    return f"Coverage {'verified' if claim['coverage_verified'] else 'not verified'}."


**Fraud Screener.** Adds points for three classic red flags: short time since policy inception, multiple prior claims, and late reporting. A score above 40 escalates to the deeper fraud investigation pattern (see Lab 2).

In [ ]:
async def screen_for_fraud(ctx: Context) -> str:
    state = await ctx.store.get('state')
    claim, policy = state['claim'], state.get('policy', {})
    flags, score = [], 0
    dol, inc = claim.get('date_of_loss', ''), policy.get('inception_date', '')
    if dol and inc and days_between(inc, dol) < 30:
        flags.append(f'Claim filed {days_between(inc, dol)} days after inception')
        score += 25
    if len(state.get('claims_history', [])) >= 3:
        flags.append(f"{len(state['claims_history'])} prior claims")
        score += 20
    if dol and days_between(dol, state.get('fnol_date', dol)) > 30:
        flags.append('Late reporting (>30 days)')
        score += 15
    score = min(score, 100)
    claim['fraud_score'] = score
    claim['red_flags'] = flags
    claim['fraud_recommendation'] = 'investigate' if score > 40 else 'proceed'
    claim['status'] = 'fraud_screening'
    log_audit(state, 'FraudScreener', f'Score {score}/100')
    await ctx.store.set('state', state)
    return f'Fraud score: {score}/100.'

**Assessment Agent.** Estimates damages from the parsed description (a vision model would do better in production), applies the deductible, and caps at the coverage limit.

In [ ]:
async def assess_damage(ctx: Context) -> str:
    state = await ctx.store.get('state')
    claim = state['claim']
    desc = claim.get('damage_description', '').lower()
    deductible, limit = claim.get('deductible', 500), claim.get('applicable_limit', 50_000)
    base = 2_000
    if 'bumper' in desc: base = 3_500
    if 'tail light' in desc or 'headlight' in desc: base += 800
    if 'airbag' in desc: base += 5_000
    if 'total' in desc: base = limit * 0.8
    payout = max(0, min(base - deductible, limit))
    claim['estimated_payout'] = payout
    claim['assessment_breakdown'] = (
        f'Repair: ${base:,.0f}. Deductible: ${deductible:,.0f}. '
        f'Limit: ${limit:,.0f}. Net: ${payout:,.0f}.'
    )
    claim['status'] = 'assessment'
    log_audit(state, 'AssessmentAgent', f'Payout ${payout:,.0f}')
    await ctx.store.set('state', state)
    return f'Estimated payout ${payout:,.0f}.'

**Decision Agent.** Three-way decision: DENIED if coverage failed, ESCALATED if fraud score crossed the threshold, otherwise APPROVED. Reasoning is verbose by design — it must reconstruct the entire chain for audit.

In [ ]:
async def make_decision(ctx: Context) -> str:
    """Make final claim decision with reasoning.

    Used by: DecisionAgent.
    Reads: coverage, fraud, assessment data from claim.
    Writes: decision, decision_reasoning to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]

    # Decision logic
    evidence_codes, evidence_complete = coverage_evidence(claim, state.get("policy", {}))
    reason_codes = []
    if not claim.get("coverage_verified", False):
        if evidence_complete and evidence_codes:
            decision = "DENIED"
            reason_codes = evidence_codes
            reasoning = (
                f"Claim denied on recorded coverage evidence: {evidence_codes}. "
                f"{claim.get('coverage_reasoning', '')}"
            )
        else:
            decision = "ESCALATED"
            reason_codes = ["INSUFFICIENT_COVERAGE_EVIDENCE"]
            reasoning = "Coverage could not be established from the supplied evidence; refer for human review."
    elif claim.get("fraud_recommendation") == "investigate":
        decision = "ESCALATED"
        reason_codes = ["FRAUD_REVIEW_REQUIRED"]
        reasoning = (
            f"Claim escalated for fraud investigation. "
            f"Fraud score: {claim.get('fraud_score', 0)}/100. "
            f"Red flags: {claim.get('red_flags', [])}."
        )
    else:
        decision = "APPROVED"
        reason_codes = ["COVERAGE_VERIFIED"]
        reasoning = (
            f"Claim approved. Coverage verified under "
            f"{claim.get('coverage_reasoning', 'policy')}. "
            f"Fraud score: {claim.get('fraud_score', 0)}/100 (low risk). "
            f"Estimated payout: ${claim.get('estimated_payout', 0):,.0f}."
        )

    claim["decision"] = decision
    claim["decision_reasoning"] = reasoning
    claim["decision_reason_codes"] = reason_codes
    claim["status"] = decision.lower()

    log_audit(state, "DecisionAgent",
              f"Decision: {decision}", reasoning)
    await ctx.store.set("state", state)
    return f"Decision: {decision}."


**Compliance Agent (guardrail).** Validates the decision against three regulatory checks: the denial reason must be valid; the reasoning must be long enough to audit; and every required pipeline field must be populated. If any check fails, the decision is BLOCKED.

In [ ]:
async def validate_compliance(ctx: Context) -> str:
    """Apply the lab's evidence checks; not a legal compliance certification.

    Used by: ComplianceAgent (guardrail — has veto power).
    Reads: decision, reasoning from claim.
    Writes: status update, compliance issues if found.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]
    issues = []

    # Check 1: Denial reason validity
    if claim.get("decision") == "DENIED":
        evidence_codes, complete = coverage_evidence(claim, state.get("policy", {}))
        codes = claim.get("decision_reason_codes", [])
        if (not complete or not evidence_codes or not isinstance(codes, list)
                or not all(isinstance(code, str) for code in codes)
                or set(codes) != set(evidence_codes)):
            issues.append("Denial requires reason codes supported by the policy and loss evidence.")
        if claim.get("coverage_verified") is not False:
            issues.append("Denial contradicts the recorded coverage result.")

    if claim.get("decision") not in {"APPROVED", "DENIED", "ESCALATED"}:
        issues.append("Unrecognized decision value.")
    if claim.get("decision") == "APPROVED":
        codes, complete = coverage_evidence(claim, state.get("policy", {}))
        if not complete or codes or claim.get("coverage_verified") is not True:
            issues.append("Approval requires complete, consistent coverage evidence.")
        if claim.get("fraud_recommendation") == "investigate":
            issues.append("Approval cannot bypass a fraud referral.")

    # Check 2: Reasoning completeness
    if len(claim.get("decision_reasoning", "")) < 50:
        issues.append(
            "Decision reasoning is missing or too short for the lab's documentation check."
        )

    # Check 3: All pipeline stages completed
    required_fields = [
        "coverage_verified", "fraud_score", "estimated_payout",
    ]
    for field in required_fields:
        if claim.get(field) is None:
            issues.append(f"Missing required field: {field}")

    if issues:
        claim["status"] = "compliance_review"
        claim["compliance_issues"] = issues
        log_audit(state, "ComplianceAgent",
                  "BLOCKED — compliance issues", str(issues))
        await ctx.store.set("state", state)
        return f"BLOCKED: {issues}"
    else:
        claim["compliance_issues"] = []
        log_audit(state, "ComplianceAgent",
                  "PASSED — lab evidence checks")
        await ctx.store.set("state", state)
        return "Decision validated against the lab's evidence checks."


## 5. Define the seven pipeline agents

Each agent's `system_prompt` is a tight job description; `can_handoff_to` names the **single next agent** in the pipeline. The Compliance agent is the terminus: it writes the final summary and stops. It *can* hand back to FNOL, but only for the rare case of a fundamentally malformed claim record — its prompt forbids routine handoffs, which would otherwise loop the pipeline forever.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

fnol_agent = FunctionAgent(
    name='FNOLAgent', description='Process first notice of loss.',
    system_prompt='Call your tool exactly once; never call it a second time. You handle initial claim intake: call process_fnol with the claim data, then hand off to DocumentParser.',
    llm=llm, tools=[process_fnol], can_handoff_to=['DocumentParser'],
)
parser_agent = FunctionAgent(
    name='DocumentParser', description='Parse submitted documents.',
    system_prompt='Call your tool exactly once; never call it a second time. Parse the submitted documents with parse_documents, then hand off to CoverageAgent.',
    llm=llm, tools=[parse_documents], can_handoff_to=['CoverageAgent'],
)
coverage_agent = FunctionAgent(
    name='CoverageAgent', description='Verify policy coverage.',
    system_prompt='Call your tool exactly once; never call it a second time. Verify coverage with verify_coverage, then hand off to FraudScreener.',
    llm=llm, tools=[verify_coverage], can_handoff_to=['FraudScreener'],
)
fraud_agent = FunctionAgent(
    name='FraudScreener', description='Score for fraud indicators.',
    system_prompt='Call your tool exactly once; never call it a second time. Score the claim with screen_for_fraud, then hand off to AssessmentAgent.',
    llm=llm, tools=[screen_for_fraud], can_handoff_to=['AssessmentAgent'],
)
assessment_agent = FunctionAgent(
    name='AssessmentAgent', description='Estimate the payout.',
    system_prompt='Call your tool exactly once; never call it a second time. Estimate the payout with assess_damage, then hand off to DecisionAgent.',
    llm=llm, tools=[assess_damage], can_handoff_to=['DecisionAgent'],
)
decision_agent = FunctionAgent(
    name='DecisionAgent', description='Make the final decision.',
    system_prompt='Call your tool exactly once; never call it a second time. Make the decision with make_decision, then hand off to ComplianceAgent.',
    llm=llm, tools=[make_decision], can_handoff_to=['ComplianceAgent'],
)
compliance_agent = FunctionAgent(
    name='ComplianceAgent', description='Regulatory guardrail with veto power.',
    system_prompt=(
        'Call your tool exactly once; never call it a second time. Validate the decision with validate_compliance. You have VETO '
        'power. Whether the result is compliant or BLOCKED, do NOT hand off: write a '
        'short final summary of the claim decision (or the blocking issues) and stop. '
        'Hand off to FNOLAgent only if the tool reports that the claim record itself '
        'is fundamentally malformed.'
    ),
    llm=llm, tools=[validate_compliance], can_handoff_to=['FNOLAgent'],
)

## 6. Assemble and run the workflow

Initialise the workflow's shared state with the sample policy, documents, and FNOL date from `common.py`. Then send the sample claim through and stream events to watch each agent take its turn.

In [ ]:
import json
from llama_index.core.agent.workflow import AgentWorkflow

workflow = AgentWorkflow(
    agents=[fnol_agent, parser_agent, coverage_agent, fraud_agent,
            assessment_agent, decision_agent, compliance_agent],
    root_agent='FNOLAgent',
    initial_state={
        'claim': {}, 'policy': SAMPLE_POLICY,
        'submitted_documents': SAMPLE_DOCUMENTS,
        'claims_history': [],
        'fnol_date': '2026-04-08',
    },
)

# Seven agents each need ~3 workflow iterations (tool call, handoff, response),
# so the default max_iterations=20 is too tight for this pipeline.
handler = workflow.run(user_msg=json.dumps(SAMPLE_CLAIM), max_iterations=60)

async for event in handler.stream_events():
    if hasattr(event, 'current_agent_name'):
        print(f'\n[Active: {event.current_agent_name}]')

result = await handler
print('\nFinal result:\n' + str(result))

## 7. Inspect the audit trail

The audit log is the single most important artifact of this pipeline. If the claim is later disputed or audited, this log must reconstruct every decision the system made.

We pull the final claim record out of the workflow's context. In LlamaIndex AgentWorkflow the context is available on the handler *after* the workflow has finished — we access it with `handler.ctx.store.get(...)`.

In [ ]:
final_state = await handler.ctx.store.get('state')
claim = final_state.get('claim', {}) if final_state else {}

print(f'Claim ID: {claim.get("claim_id")}')
print(f'Decision: {claim.get("decision")}')
print(f'Payout:   ${claim.get("estimated_payout", 0):,.0f}')
print(f'Status:   {claim.get("status")}\n')
print('=' * 50)
print('  AUDIT TRAIL')
print('=' * 50)
for entry in claim.get('audit_log', []):
    print(f'  {entry["timestamp"]} [{entry["agent"]}] {entry["action"]}')
    if entry.get('detail'):
        print(f'      {entry["detail"][:120]}')

## 8. Results

You processed a real-world auto-collision claim through seven specialist agents and produced a fully auditable decision. **What to notice about the claims-pipeline architecture:**

* **Compliance is an agent, not an afterthought.** A guardrail agent with veto power gives you a   single, testable place to enforce regulation.
* **Audit trails are written by tools, not by agents.** Every tool calls `log_audit` as it   mutates state — the trail is a byproduct, not extra work.
* **Pydantic models pay off.** `ClaimRecord`, `PolicyRecord`, and friends keep the contract   between agents explicit; a mismatch surfaces at the next agent's tool call.
* **The same architecture extends to property, health, or liability claims** by swapping the tools,   not the agent topology.